# Неделя 4 — Итоговый отчёт

Задание: `docs/week4_experiment.md`

## 1. Постановка задачи и определение оттока

Задача: построить ранжирование клиентов по риску оттока, чтобы бизнес мог
проактивно работать с теми, кто вот-вот уйдёт.

Отток в модельном датасете: клиент теряет выручку в окне прогноза (3 месяца)
после зазора в 2 месяца от снапшота. В метке учтены два типа:
- explicit — договор расторгнут;
- silent — договор жив, но выручка за последние 3 месяца < 20% от нормы клиента
  (найдено на неделе 1: 1546 таких клиентов).

Модель предсказывает не "уйдёт/не уйдёт", а ранжирует: кого обзванивать в первую очередь.

## 2. Данные: объём, период, доля оттока по сегментам

Данные: справочник клиентов + помесячное потребление. Ниже — живые цифры.

In [ ]:
import pandas as pd
clients = pd.read_parquet('../data/processed/clients.parquet')
usage = pd.read_parquet('../data/processed/usage.parquet')
dataset = pd.read_parquet('../data/processed/dataset.parquet')

print('Клиентов:', len(clients))
print('Строк usage:', len(usage))
print('Период usage:', usage['month'].min(), '...', usage['month'].max())

test = dataset[dataset['snapshot_date'] == '2025-10']
print('\nДоля оттока в test по сегментам:')
print(test.groupby('segment')['target'].mean().round(4))

## 3. Схема сборки выборки (snapshot'ы)

Одна строка выборки = клиент в точке снапшота.
- Наблюдение: 6 месяцев по месяц снапшота включительно -> признаки
- Зазор: 2 месяца -> не используются (иначе модель "читает настоящее":
  коллапс выручки начинается за 4-5 мес до ухода и попадал бы в признаки)
- Прогноз: 3 месяца после зазора -> таргет

Снапшоты: 2024-07, 2024-10, 2025-01, 2025-04, 2025-07, 2025-10.
train = 2024-07, 2024-10; val = 2025-01; test = 2025-10.
Пересечения месяцев между train и test нет (train заканчивается на 2025-03,
test начинается с 2025-05).
Дополнительно исключены клиенты с историей < 4 месяцев в окне наблюдения.

## 4. Признаки: группы и интуиция

18 числовых + 3 категориальных признака, пять групп:
- платежи: среднее/мин/макс/последняя выручка, отношение последняя/среднее
- динамика: тренд, волатильность, месяцы подряд снижения
- объём отношений: число SIM, срок жизни клиента (tenure_months)
- сигналы боли: тикеты в поддержку, задолженность
- категориальные: сегмент, продукт, регион

Интуиция из недели 1 (event study): клиент затухает за 3-4 месяца до ухода
(выручка 0.95 -> 0.56 -> 0.21 от нормы), а тикеты растут с 0.27 до 1.86.
Признаки платежей и динамики ловят ровно этот паттерн.

## 5. Схема валидации и метрики

Валидация out-of-time: train/val/test разнесены по времени, тест модель не видит.
Метрики: ROC-AUC, PR-AUC, Precision@10%, Lift@10%.
Accuracy не используется: отток в test ~8%, и модель "никто не уйдёт" дала бы
92% accuracy при нулевой пользе. Нам важно ранжирование, а не угадывание класса.

## 6. Результаты моделей: правило → логрег → CatBoost

| Модель | ROC-AUC | Lift@10% |
|---|---|---|
| Правило (падение выручки) | 0.793 | 5.7x |
| Логистическая регрессия | 0.827 | 6.0x |
| CatBoost | 0.843 | 6.5x |

Порядок правильный: каждая следующая модель сильнее. CatBoost останавливается
на ~135 итерации из 1000 — задача честная, без переобучения.

## 7. Эксперимент A/B/C: ответ на главный вопрос проекта

| Подход | ROC-AUC | Lift@10% |
|---|---|---|
| A: единая без segment | 0.841 | 6.5x |
| B: единая + segment | 0.843 | 6.5x |
| C: отдельные по сегментам | 0.830 | 6.5x |

По сегментам (AUC):
| Сегмент | A | B | C |
|---|---|---|---|
| крупный | 0.837 | 0.840 | 0.826 |
| средний | 0.844 | 0.850 | 0.850 |
| малый | 0.826 | 0.827 | 0.833 |

Bootstrap (1000 выборок): B лучше A в 96% (+0.003), B лучше C в 100% (+0.014),
A лучше C в 100% (+0.011).

Вывод по шаблону: подход B показал AUC 0.843, lift 6.5x; подход A — 0.841,
различие значимо, но крошечное (+0.003); подход C — 0.830, значимо хуже.
Рекомендуем единую модель с сегментом как признаком (B). Отдельные модели по
сегментам не нужны: на крупном бизнесе у C мало данных (389 оттоков в train,
остановка на 15 итерации), и он теряет общие паттерны. Ограничение: выигрыш
B над A мал, то есть сам по себе сегмент почти не несёт информации —
работает динамика выручки, одинаковая во всех сегментах.

## 8. Интерпретация: топ-факторы оттока, примеры клиентов

Топ-факторы по SHAP (все с бизнес-смыслом, никаких технических ID):
1. revenue_last_to_mean (34%) — выручка упала относительно своей нормы -> риск
2. revenue_declining_months (12%) — месяцы подряд снижения
3. revenue_min_6m, revenue_trend, revenue_volatility — глубина и нестабильность падения
4. debt_last — текущий долг
5. tickets_mean_6m — жалобы в поддержку

Пример 1 (ушёл, score 0.99): средний бизнес, выручка 56% от нормы,
3 месяца снижения, долг 1870 руб — все красные факторы толкают прогноз вверх.
Пример 2 (остался, score 0.00): малый бизнес, выручка 162% от нормы (растёт),
нет снижения и долга — прогноз вниз.

## 9. Бизнес-эффект: lift, эффект от обзвона топ-10%

В test 36 123 клиента, отток ~8% (~2 900 уходов).
Обзвон топ-10% по риску (3 600 клиентов) ловит ~65% всех будущих уходов
(~1 900 клиентов), lift = 6.5x.
Для сравнения: случайный обзвон тех же 3 600 поймал бы ~10% (~290 уходов).
То есть отдел удержания работает по списку из 3 600 клиентов вместо перебора
всей базы и ловит в 6.5 раза больше уходов.

## 10. Ограничения и что делать дальше

Ограничения:
- train всего из двух снапшотов (~60k строк) — мало данных для сложных моделей;
- крупный бизнес: мало оттоков, отдельная модель там не обучается;
- silent-отток — эвристика (порог 20% от нормы), часть "тихих" может быть
  сезонностью или паузой, а не уходом;
- сбой марта 2025 (тикеты 1.78, трафик -30%) попадает в окна наблюдения части
  снапшотов и добавляет шум в признаки тикетов;
- не использованы группы компаний (19 360 названий на 50 400 клиентов):
  синхронный уход филиалов может быть отдельным признаком.

Что дальше:
- признаки тренда тикетов и групп компаний;
- оценка в деньгах: удержанный клиент = LTV, сравнить стоимость обзвона и пользу;
- калибровка вероятностей для порога обзвона;
- регулярное дообучение по мере поступления новых месяцев.